# tensor + data parallel groups

核心职责是：构建用于处理复杂混合并行策略（张量并行 TP + 数据并行 DP + 上下文并行 CP）的通信组。

在大模型训练中，为了极致压榨硬件性能，框架往往会同时使用多种并行策略。这段代码就是为了给这些策略“画圈子”，明确哪些 GPU 需要凑在一起进行特定的通信。

下面为你逐层拆解这段代码的逻辑：
🎯 1. 核心目标：构建多维度的并行通信组
代码中主要创建了三类通信组，它们分别服务于不同的并行策略组合：
- _TENSOR_AND_DATA_PARALLEL_GROUP_WITH_CP：融合了 TP + DP + CP 的超大通信组。
- _TENSOR_AND_DATA_PARALLEL_GROUP：基础的 TP + DP 通信组（不考虑上下文并行）。
- _TENSOR_AND_CONTEXT_PARALLEL_GROUP：融合了 TP + CP 的通信组。

In [ ]:
    '''
    张量 数据并行组：
    '''
    # Build the tensor + data parallel groups.
    global _TENSOR_AND_DATA_PARALLEL_GROUP
    global _TENSOR_AND_DATA_PARALLEL_GROUP_WITH_CP
    assert (
        _TENSOR_AND_DATA_PARALLEL_GROUP is None
    ), 'Tensor + data parallel group is already initialized'
    '''
    构建 TP+DP+CP 混合组:
    '''
    for ranks in decoder_rank_generator.get_ranks('tp-dp-cp'):
        group = create_group(
            ranks,
            timeout=timeout,
            pg_options=get_nccl_options("tp_dp_cp", nccl_comm_cfgs),
            group_desc="TENSOR_AND_DATA_PARALLEL_GROUP_WITH_CP",
        )
        '''认领组：'''
        if rank in ranks:
            _TENSOR_AND_DATA_PARALLEL_GROUP_WITH_CP = group
    '''
    构建基础 TP+DP 组
    '''        
    for ranks in decoder_rank_generator.get_ranks('tp-dp'):
        group = create_group(
            ranks,
            timeout=timeout,
            pg_options=get_nccl_options("tp_dp", nccl_comm_cfgs),
            group_desc="TENSOR_AND_DATA_PARALLEL_GROUP",
        )
        '''认领组：'''
        if rank in ranks:
            _TENSOR_AND_DATA_PARALLEL_GROUP = group

    global _TENSOR_AND_CONTEXT_PARALLEL_GROUP
    assert (
        _TENSOR_AND_CONTEXT_PARALLEL_GROUP is None
    ), 'Tensor + context parallel group is already initialized'
    
    '''
    构建 TP+CP 组
    '''
    for ranks in decoder_rank_generator.get_ranks('tp-cp'):
        group = create_group(
            ranks,
            timeout=timeout,
            pg_options=get_nccl_options("tp_cp", nccl_comm_cfgs),
            group_desc="TENSOR_AND_CONTEXT_PARALLEL_GROUP",
        )
        '''认领组：'''
        if rank in ranks:
            _TENSOR_AND_CONTEXT_PARALLEL_GROUP = group


## 💡 3. 为什么需要这些混合组？（结合之前的知识）
结合你之前了解的流水线并行（PP）和 Embedding 组，我们可以把大模型的并行策略看作一个立体的“魔方”：
- TP（张量并行）：负责把单个算子（如矩阵乘法）切分到多张卡上计算。
- DP（数据并行）：负责把不同的数据批次（Batch）分给不同的卡。
- CP（上下文并行）：通常用于处理超长序列，把序列的上下文切分到不同卡上。
- PP（流水线并行）：负责把模型的不同层（Layer）分给不同的卡。

这段代码就是在为这些维度的组合“画圈子”。比如 _TENSOR_AND_DATA_PARALLEL_GROUP 圈定了一个范围，在这个范围内的 GPU 需要共同完成 TP 的算子切分通信和 DP 的数据同步通信。
## 📌 总结
- 这段代码是分布式训练框架中“通信拓扑的构建者”。
- 它通过 decoder_rank_generator 获取底层的并行策略划分，然后为 TP、DP、CP 的不同组合创建专属的通信组。
- 这些组是后续进行梯度同步、参数更新和激活值传递的基础设施，确保了复杂混合并行策略能够高效、正确地执行。- 